In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions

data_dir = r"D:\ImageNet"
img_size = (224, 224)   
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="training", seed=42,
    image_size=img_size, batch_size=batch_size
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir, validation_split=0.2, subset="validation", seed=42,
    image_size=img_size, batch_size=batch_size
)

class_names = train_ds.class_names
print("Classes:", class_names)

model = ResNet50(weights="imagenet")
images, labels = next(iter(test_ds))
images = preprocess_input(images.numpy())

predictions = model.predict(images)
top5 = decode_predictions(predictions, top=5)

for i in range(min(10, len(images))):
    plt.figure(figsize=(5, 4))
    plt.imshow(np.clip((images[i] + [103.939, 116.779, 123.68])[..., ::-1] / 255, 0, 1))
    plt.title(f"Original: {class_names[labels[i]]}\nPredicted: {top5[i][0][1]}")
    plt.axis("off")
    plt.show()

    print("Top-5 predictions:")
    for _, name, score in top5[i]:
        print(f"{name}: {score:.4f}")

    print("Input shape:", images[i].shape)
    print("Preprocessed shape:", images[i:i+1].shape)
    print("Prediction shape:", predictions[i].shape)

layer_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("conv3_block4_out").output
)
features = layer_model.predict(images[:1])

plt.figure(figsize=(12, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(features[0, :, :, i], cmap="viridis")
    plt.axis("off")
plt.show()

print("Intermediate feature map shape:", features.shape)

In [ ]:

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions

data_path = "D:/ImageNet/Images"
img_size = (224, 224)

random.seed(42)
classes = [d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))]
classes = random.sample(classes, min(10, len(classes)))

images, labels = [], []

for c in classes:
    folder = os.path.join(data_path, c)
    files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))]
    if files:
        file = random.choice(files)
        img = load_img(os.path.join(folder, file), target_size=img_size)
        images.append(img_to_array(img))
        labels.append(c)

x = np.array(images, dtype=np.float32)
x_processed = preprocess_input(x.copy())

model = ResNet50(weights="imagenet")
feature_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("conv3_block4_out").output
)

predictions = model.predict(x_processed, verbose=0)
features = feature_model.predict(x_processed, verbose=0)

print("Number of images:", len(images))
print("Original image shape:", x.shape)
print("Preprocessed image shape:", x_processed.shape)
print("Prediction output shape:", predictions.shape)
print("Intermediate feature map shape:", features.shape)
print("Preprocessing: resize to 224x224, convert to array, ResNet50 preprocessing")

plt.figure(figsize=(15, 6))
for i in range(len(images)):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x[i].astype("uint8"))
    plt.title("Original: " + labels[i])
    plt.axis("off")
plt.tight_layout()
plt.show()

for i in range(len(images)):
    top5 = decode_predictions(predictions[i:i+1], top=5)[0]
    print("\nOriginal folder label:", labels[i])
    print("Top-5 predictions:")
    for synset, name, score in top5:
        print(f"{name}: {score:.4f} ({score*100:.2f}%)")
    print("Top-1 prediction matches original label:", top5[0][0] == labels[i])

plt.figure(figsize=(15, 5))
for i in range(min(5, len(images))):
    plt.subplot(1, 5, i + 1)
    plt.imshow(features[i, :, :, 0], cmap="viridis")
    plt.title("Feature map " + str(i + 1))
    plt.axis("off")
plt.tight_layout()
plt.show()
